In [ ]:
using CUDA

# List all GPUs – CUDA.devices() returns a vector of CuDevice objects
for (i, dev) in enumerate(CUDA.devices())
    println("$i: $(CUDA.name(dev))")
end

In [1]:
using NeuroDSL   # sans include

In [ ]:
using CUDA, NeuroDSL

const _Backend = NeuroDSL.Backend
devices = collect(CUDA.devices())
@assert length(devices) >= 2

function build_graph(gpu_idx, ns)
    CUDA.device!(gpu_idx - 1)
    g = NeuroGraph(; namespace=ns, device=_Backend.active_device())
    set!(g, :x, _Backend.rand32(g.device, 64, 128); is_param=false, namespace=ns)
    set!(g, :W, _Backend.rand32(g.device, 64, 128); is_param=true, namespace=ns)
    addrule!(g, GraphRule(:y, [:x, :W], :matmul; attrs=Dict(:trans_b=>true), namespace=ns))
    set!(g, :target, _Backend.rand32(g.device, 64, 64); is_param=false, namespace=ns)
    addrule!(g, GraphRule(:loss, [:y, :target], :mse_loss; namespace=ns))
    return g
end

g1 = build_graph(1, :gpu1)
g2 = build_graph(2, :gpu2)

ctx1, ctx2 = CtxStore(), CtxStore()

for (g, ctx, id) in [(g1, ctx1, 1), (g2, ctx2, 2)]
    CUDA.device!(id - 1)
    demand!(g, :loss; ctx_store=ctx)
    println("GPU $id loss = ", only(node(g, :loss).value))
    backward_graph!(g, :loss; ctx_store=ctx)
    println("Gradient norm = ", norm(node(g, :W).gradient))
end

GPU 1 loss = 981.9281
Gradient norm = 44.075245
GPU 2 loss = 967.8411
Gradient norm = 43.67984


In [ ]:
using CUDA, NeuroDSL

const B = NeuroDSL.Backend

function run_on_gpu(gpu_id, ns, ctx)
    CUDA.device!(gpu_id)
    g = NeuroGraph(namespace=ns, device=B.active_device())
    
    x_val = B.rand32(g.device, 64, 128)
    w_val = B.rand32(g.device, 64, 128)
    set!(g, :x, x_val; is_param=false, namespace=ns)
    set!(g, :W, w_val; is_param=true, namespace=ns)
    
    addrule!(g, GraphRule(:y, [:x, :W], :matmul;
              attrs=Dict(:trans_b=>true), namespace=ns))
    
    target_val = B.rand32(g.device, 64, 64)
    set!(g, :target, target_val; is_param=false, namespace=ns)
    addrule!(g, GraphRule(:loss, [:y, :target], :mse_loss; namespace=ns))
    
    demand!(g, :loss; ctx_store=ctx)
    backward_graph!(g, :loss; ctx_store=ctx)
    
    return node(g, :loss).value, node(g, :W).gradient
end

# Exécution sur les deux GPUs
loss1, grad1 = run_on_gpu(0, :gpu0, CtxStore())
loss2, grad2 = run_on_gpu(1, :gpu1, CtxStore())

# Copie CPU pour éviter les erreurs de contexte
grad1_cpu = Array(grad1)
grad2_cpu = Array(grad2)

println("GPU0 loss = $(only(loss1)), grad norm = $(norm(grad1_cpu))")
println("GPU1 loss = $(only(loss2)), grad norm = $(norm(grad2_cpu))")

GPU0 loss = 970.6022, grad norm = 43.800556
GPU1 loss = 1024.5609, grad norm = 45.886787


In [ ]:
using CUDA, NeuroDSL
const B = NeuroDSL.Backend

function run_on_gpu(gpu_id, ns, ctx)
    CUDA.device!(gpu_id)
    g = NeuroGraph(namespace=ns, device=B.active_device())
    x = B.rand32(g.device, 64, 128); set!(g, :x, x; is_param=false, namespace=ns)
    w = B.rand32(g.device, 64, 128); set!(g, :W, w; is_param=true, namespace=ns)
    addrule!(g, GraphRule(:y, [:x,:W], :matmul; attrs=Dict(:trans_b=>true), namespace=ns))
    t = B.rand32(g.device, 64, 64); set!(g, :target, t; is_param=false, namespace=ns)
    addrule!(g, GraphRule(:loss, [:y,:target], :mse_loss; namespace=ns))
    demand!(g, :loss; ctx_store=ctx)
    backward_graph!(g, :loss; ctx_store=ctx)
    return node(g, :loss).value, node(g, :W).gradient
end

loss1, grad1 = run_on_gpu(0, :gpu0, CtxStore())
loss2, grad2 = run_on_gpu(1, :gpu1, CtxStore())
println("GPU0 loss = $(only(loss1)), grad norm = $(norm(Array(grad1)))")
println("GPU1 loss = $(only(loss2)), grad norm = $(norm(Array(grad2)))")

GPU0 loss = 998.0891, grad norm = 44.70354
GPU1 loss = 984.4751, grad norm = 44.398674
